# Active learning для сегментації

## Part 2. Uncertainty на синтетичних прогнозах

Перевіряємо margin uncertainty та predictive entropy без датасету, encoder і навчання. Коли з'явиться segmentation head, замінимо синтетичні logits його прогнозами. Функції не отримують GT masks або Oracle.

Логіка margin uncertainty відповідає [лекційному notebook, розділ 5](https://github.com/AI220-Computer-Vision/kse_advanced_computer_vision/blob/main/Module_2/Lecture_2/active_learning_moons.ipynb). Тут обчислюємо її для кожного пікселя у PyTorch, а потім усереднюємо по зображенню.

Цей блок працює на CPU. Потрібен лише PyTorch; завантажень, GPU-запусків та зовнішнього логування немає.

In [1]:
import math
from collections.abc import Iterable
from typing import Literal

import torch
from torch import Tensor

NUM_CLASSES = 21
IGNORE_INDEX = 255
SEED = 42
SIGMA_FLOOR = 1e-6
ECE_BINS = 5
TEMPERATURE_GRID = (0.25, 0.5, 1.0, 2.0, 4.0, 8.0)

### Від logits до pixel uncertainty

Вхід `logits` має форму `[B, C, H, W]` і dtype `float32`. Softmax застосовуємо по класах, тобто `dim=1`. У VOC `C=21`, включно з фоном. Вихід має форму `[B, H, W]`.

- **Margin uncertainty:** $U=1-(p_{(1)}-p_{(2)})$, де $p_{(1)}$ та $p_{(2)}$ — дві найбільші ймовірності. Близькі значення дають високу uncertainty.
- **Normalized predictive entropy:** $U=-\sum_{c=1}^{C}p_c\log p_c / \log C$. Враховує весь розподіл; рівномірний розподіл дає 1.

Обидва scores лежать у `[0, 1]`, але однакове число не означає однаковий зміст. Якщо два класи мають імовірності приблизно 0.5, margin uncertainty близька до 1, а entropy при 21 класі близька до $\log 2 / \log 21 \approx 0.228$.

`torch.special.entr` обчислює $-p\log p$ з визначеним значенням 0 при `p=0`. Температуру поки не підбираємо: використовуємо початкові logits, тобто `tau=1`. Жоден із цих scores сам по собі не розділяє epistemic та aleatoric uncertainty.

In [2]:
@torch.no_grad()
def pixel_uncertainty(
    logits: Tensor, method: Literal["margin", "entropy"]
) -> Tensor:
    """Перетворює raw logits [B, C, H, W] на uncertainty [B, H, W]."""
    if logits.ndim != 4 or min(logits.shape) < 1 or logits.shape[1] < 2:
        raise ValueError("Очікуємо непорожні logits [B, C, H, W], C >= 2")
    if logits.dtype != torch.float32 or not torch.isfinite(logits).all():
        raise ValueError("logits повинні мати dtype float32 без NaN та infinity")
    if method not in ("margin", "entropy"):
        raise ValueError("method має бути 'margin' або 'entropy'")

    probabilities = logits.softmax(dim=1)
    if method == "margin":
        top_two = probabilities.topk(k=2, dim=1).values
        uncertainty = 1.0 - (top_two[:, 0] - top_two[:, 1])
    else:
        uncertainty = torch.special.entr(probabilities).sum(dim=1)
        uncertainty = uncertainty / math.log(logits.shape[1])

    # Округлення float32 може дати значення трохи за теоретичними межами.
    return uncertainty.clamp(0.0, 1.0)

### Один score на зображення

Початкова агрегація — середня uncertainty по геометрично валідних пікселях. Ділимо на їхню кількість **окремо для кожного зображення**, а не на розмір усього batch.

`valid_pixels: bool [B, H, W]` описує лише геометрію, наприклад відсутність padding. Не можна будувати її з `GT != 255` для нерозміченого pool: це читання прихованої розмітки. Якщо padding немає, маска складається з `True`.

Недолік середнього: великий упевнений фон може зменшувати score зображення з малим невизначеним об'єктом. Поки фіксуємо цю агрегацію, а не підбираємо її за official val.

Функція зберігає порядок batch. Для реальних прогнозів передаватимемо `(batch.ids, image_scores)` у наявний `assemble_scores`, щоб відновити порядок pool IDs.

In [3]:
@torch.no_grad()
def mean_image_uncertainty(
    pixel_scores: Tensor, valid_pixels: Tensor
) -> Tensor:
    """Усереднює [B, H, W] до [B], виключаючи лише geometry padding."""
    if pixel_scores.ndim != 3 or min(pixel_scores.shape) < 1:
        raise ValueError("Очікуємо непорожні pixel_scores [B, H, W]")
    if pixel_scores.dtype != torch.float32 or not torch.isfinite(pixel_scores).all():
        raise ValueError("pixel_scores повинні мати dtype float32 без NaN та infinity")
    if ((pixel_scores < 0) | (pixel_scores > 1)).any():
        raise ValueError("pixel_scores повинні лежати в [0, 1]")
    if valid_pixels.dtype != torch.bool or valid_pixels.shape != pixel_scores.shape:
        raise ValueError("valid_pixels повинна бути bool-маскою тієї самої форми")
    if valid_pixels.device != pixel_scores.device:
        raise ValueError("Scores і geometry mask мають бути на одному device")

    counts = valid_pixels.sum(dim=(1, 2))
    if (counts == 0).any():
        raise ValueError("Кожне зображення повинно мати хоча б один валідний піксель")
    totals = pixel_scores.masked_fill(~valid_pixels, 0.0).sum(dim=(1, 2))
    return totals / counts

### Три контрольовані прогнози

Створимо три зображення по 2×3 пікселі. Усередині кожного прогноз однаковий у всіх пікселях, щоб легко перевірити відповідь. Це штучні logits, а не результати навченої моделі.

1. Один клас значно переважає: обидва scores близькі до 0.
2. Усі 21 класи рівноймовірні: обидва scores дорівнюють 1.
3. Два класи однаково правдоподібні: margin близька до 1, entropy близька до 0.228.

In [4]:
example_logits = torch.full((3, NUM_CLASSES, 2, 3), -12.0)
example_logits[0, 0] = 12.0
example_logits[1] = 0.0
example_logits[2, :2] = 12.0
example_valid_pixels = torch.ones((3, 2, 3), dtype=torch.bool)

example_scores = {}
for method in ("margin", "entropy"):
    pixel_scores = pixel_uncertainty(example_logits, method)
    example_scores[method] = mean_image_uncertainty(
        pixel_scores, example_valid_pixels
    )

print(f"{'Синтетичний прогноз':<28} {'margin':>10} {'entropy':>10}")
names = ("Один упевнений клас", "Рівномірний розподіл", "Два рівноймовірні класи")
for index, name in enumerate(names):
    print(
        f"{name:<28} {example_scores['margin'][index].item():10.6f}"
        f" {example_scores['entropy'][index].item():10.6f}"
    )

Синтетичний прогноз              margin    entropy
Один упевнений клас            0.000000   0.000000
Рівномірний розподіл           1.000000   1.000000
Два рівноймовірні класи        1.000000   0.227670


### Uncertainty top-B

Після агрегації маємо `scores: float32 [N]`. Індекс `scores[i]` — постійний pool ID, а не позиція зображення в поточному batch. Вибираємо `n_select` **нових** IDs лише з `unlabeled_ids`. Уже розмічені зображення не беруть участі, навіть якщо мають більшу uncertainty.

Порядок вибору: score за спаданням, а при однакових scores — менший ID. Для цього спочатку впорядковуємо IDs, потім застосовуємо `argsort(..., descending=True, stable=True)`. Звичайний `topk` не гарантує потрібного порядку при нічиїй.

`n_select=0` повертає порожній список; бюджет більший за `|U|` є помилкою. Функція не змінює вхідні дані та не звертається до Oracle. При cold start, коли навченого head ще немає, runner використає спільний Random fallback, а не вигадані uncertainty scores.

Нижче — шість синтетичних image scores. ID `3` уже labeled, тому score `1.0` не робить його кандидатом. IDs `1` і `4` мають однаковий score `0.95`; спочатку вибираємо `1`.

In [5]:
def uncertainty_top_b(
    scores: Tensor, unlabeled_ids: list[int], n_select: int
) -> list[int]:
    """Вибирає нові pool IDs за спаданням score; нічию вирішує менший ID."""
    if scores.ndim != 1 or scores.dtype != torch.float32:
        raise ValueError("Очікуємо scores: float32 [N]")
    if not torch.isfinite(scores).all() or (scores < 0).any():
        raise ValueError("Scores повинні бути скінченними та невід'ємними")
    if type(n_select) is not int or not 0 <= n_select <= len(unlabeled_ids):
        raise ValueError("n_select має бути цілим числом від 0 до розміру U")
    if any(type(i) is not int or not 0 <= i < len(scores) for i in unlabeled_ids):
        raise ValueError("Кандидати мають бути цілими pool IDs у межах scores")
    if len(set(unlabeled_ids)) != len(unlabeled_ids):
        raise ValueError("Кандидати не повинні містити повторних IDs")

    candidates = torch.tensor(sorted(unlabeled_ids), dtype=torch.long, device=scores.device)
    order = torch.argsort(scores[candidates], descending=True, stable=True)
    return candidates[order[:n_select]].tolist()

In [6]:
pool_scores = torch.tensor([0.10, 0.95, 0.80, 1.00, 0.95, 0.60])
unlabeled_ids = [0, 1, 2, 4, 5]  # ID 3 уже labeled.
selected_ids = uncertainty_top_b(pool_scores, unlabeled_ids, n_select=3)

print(f"Вибрані IDs у порядку пріоритету: {selected_ids}")
print("Їхні scores:", [round(pool_scores[i].item(), 2) for i in selected_ids])

Вибрані IDs у порядку пріоритету: [1, 4, 2]
Їхні scores: [0.95, 0.95, 0.8]


## Part 3. Gaussian kernel і MaxHerding

MaxHerding вибирає представників pool за схожістю embeddings. Прогнози head та мітки класів йому не потрібні, тому він працює і при cold start, коли `L` порожнє.

### Відстані та ширина kernel

Для реального запуску використовуємо L2-normalized image embeddings `[N, D]` від encoder. Рядки зберігають постійні pool IDs. Обчислюємо евклідові відстані один раз, потім:

$$\sigma = \operatorname{median}_{i<j}\|h_i-h_j\|_2, \qquad K_{i,j}=\exp\left(-\frac{\|h_i-h_j\|_2^2}{\sigma^2}\right).$$

Діагональ не бере участі в median, кожну пару враховуємо один раз. Для парної кількості відстаней беремо середнє двох центральних значень. Якщо median нульова через дублікати, застосовуємо floor `1e-6`. Для оцінки median потрібно хоча б два зображення.

Це label-free правило MaxHerding, а не adaptive sigma для UHerding. Його обчислимо на всьому train pool і залишимо незмінним між раундами за незмінних embeddings. Official val не використовуємо.

`cdist` рахує відстані без створення tensor `[N, N, D]`. Прямий режим обчислення зберігає нульові відстані між однаковими векторами без похибки від віднімання близьких квадратів норм. Повна kernel matrix для 1464 зображень у float32 займає приблизно 8.18 MiB; проміжні tensors потребують додаткової пам'яті.

In [7]:
@torch.no_grad()
def pairwise_distances(embeddings: Tensor) -> Tensor:
    """Евклідові відстані [N, N]; порядок рядків відповідає pool IDs."""
    if embeddings.ndim != 2 or min(embeddings.shape) == 0:
        raise ValueError("Очікуємо непорожні embeddings [N, D]")
    if embeddings.dtype != torch.float32 or not torch.isfinite(embeddings).all():
        raise ValueError("Embeddings мають бути скінченними float32")
    return torch.cdist(
        embeddings, embeddings, p=2, compute_mode="donot_use_mm_for_euclid_dist"
    )


@torch.no_grad()
def median_kernel_sigma(distances: Tensor) -> float:
    """Label-free sigma з матриці евклідових відстаней train pool."""
    if distances.shape[0] < 2:
        raise ValueError("Для median-відстані потрібно хоча б два зображення")
    rows, columns = torch.triu_indices(
        len(distances), len(distances), offset=1, device=distances.device
    )
    median = torch.quantile(distances[rows, columns], 0.5).item()
    return max(median, SIGMA_FLOOR)


@torch.no_grad()
def gaussian_kernel(distances: Tensor, sigma: float) -> Tensor:
    """Gaussian similarity [N, N] з евклідових відстаней, без множника 2."""
    if isinstance(sigma, bool) or not math.isfinite(sigma) or sigma <= 0:
        raise ValueError("sigma має бути скінченним додатним числом")
    return torch.exp(-(distances / sigma).square())

### Greedy-вибір за приростом покриття

Дотримуємося [лекційного notebook, розділ 4](https://github.com/AI220-Computer-Vision/kse_advanced_computer_vision/blob/main/Module_2/Lecture_2/active_learning_moons.ipynb). У `K` рядки — targets, стовпці — кандидати. Набір targets `U₀` фіксуємо на початку batch.

$$m_i=\max_{l\in L\cup S}K_{i,l},\qquad \Delta(c)=\sum_{i\in U_0}\max(K_{i,c}-m_i,0).$$

1. Якщо `L` порожнє, початкове покриття `m` нульове. Інакше враховуємо найкращу схожість із уже labeled зображеннями.
2. Вибираємо доступний ID із найбільшим gain. При нічиїй беремо найменший ID.
3. Оновлюємо `m = maximum(m, K[:, chosen])` і забороняємо повторний вибір цього кандидата.
4. Перераховуємо gains перед наступним вибором. Target row вибраної точки залишається в objective.

Ділення суми на незмінне `|U₀|` не змінює вибраний ID, тому в коді використовуємо суму. Навіть за нульових gains добираємо рівно `n_select` нових IDs за правилом нічиєї.

Функція отримує kernel від `gaussian_kernel`, а не embeddings, щоб не обчислювати матрицю повторно між раундами. `U` визначаємо як усі pool IDs, яких немає в `L`; маски та Oracle не передаємо.

In [8]:
@torch.no_grad()
def greedy_kernel_selection(
    kernel: Tensor,
    labeled_ids: list[int],
    n_select: int,
    weights: Tensor | None = None,
) -> list[int]:
    """Спільний greedy-цикл; weights [N] зважують targets, тобто rows."""
    if kernel.ndim != 2 or kernel.shape[0] != kernel.shape[1]:
        raise ValueError("Очікуємо квадратну kernel matrix [N, N]")
    n_pool = len(kernel)
    if any(type(i) is not int or not 0 <= i < n_pool for i in labeled_ids):
        raise ValueError("L має містити цілі IDs у межах pool")
    labeled = set(labeled_ids)
    if len(labeled) != len(labeled_ids):
        raise ValueError("L не повинно містити повторних IDs")
    unlabeled_ids = [i for i in range(n_pool) if i not in labeled]
    if type(n_select) is not int or not 0 <= n_select <= len(unlabeled_ids):
        raise ValueError("n_select має бути цілим числом від 0 до розміру U")
    if weights is not None:
        if weights.shape != (n_pool,) or weights.dtype != torch.float32:
            raise ValueError("Очікуємо weights: float32 [N] у порядку pool IDs")
        if weights.device != kernel.device or not torch.isfinite(weights).all():
            raise ValueError("Weights мають бути скінченними та на device kernel")
        if (weights < 0).any():
            raise ValueError("Weights не можуть бути від'ємними")
    if n_select == 0:
        return []

    targets = kernel[unlabeled_ids]  # [|U₀|, N], незмінні rows протягом batch.
    coverage = (
        targets[:, labeled_ids].amax(dim=1)
        if labeled_ids else kernel.new_zeros(len(unlabeled_ids))
    )
    target_weights = (
        kernel.new_ones(len(unlabeled_ids))
        if weights is None else weights[unlabeled_ids]
    )
    scale = target_weights.max()
    if scale > 0:
        target_weights = target_weights / scale  # Спільний множник не змінює argmax.

    candidates = targets[:, unlabeled_ids]  # Columns у порядку зростання pool ID.
    available = torch.ones(len(unlabeled_ids), dtype=torch.bool, device=kernel.device)
    selected_ids = []
    for _ in range(n_select):
        improvements = (candidates - coverage[:, None]).clamp_min_(0)
        gains = (improvements * target_weights[:, None]).sum(dim=0)
        gains.masked_fill_(~available, -torch.inf)
        position = gains.argmax().item()
        selected_ids.append(unlabeled_ids[position])
        coverage = torch.maximum(coverage, candidates[:, position])
        available[position] = False
    return selected_ids


def maxherding(kernel: Tensor, labeled_ids: list[int], n_select: int) -> list[int]:
    """MaxHerding використовує однакові ваги всіх targets."""
    return greedy_kernel_selection(kernel, labeled_ids, n_select)

### Чому не можна один раз взяти top-B за початковими gains

Візьмемо три групи з двома однаковими normalized embeddings у кожній: IDs `[0, 1]`, `[2, 3]`, `[4, 5]`. Дублікати тут навмисні, щоб показати надлишковий відбір.

За початковими gains найкращі IDs `2` та `3`, але вони представляють одну групу. MaxHerding спочатку вибирає `2`, оновлює покриття, а другим бере `0` з іншої групи. Якщо `2` уже labeled, його покриття враховується ще до першого нового вибору.

In [9]:
toy_embeddings = torch.tensor([
    [1.0, 0.0], [1.0, 0.0],
    [0.0, 1.0], [0.0, 1.0],
    [-1.0, 0.0], [-1.0, 0.0],
])
toy_distances = pairwise_distances(toy_embeddings)
toy_sigma = median_kernel_sigma(toy_distances)
toy_kernel = gaussian_kernel(toy_distances, toy_sigma)

one_shot_ids = torch.argsort(toy_kernel.sum(dim=0), descending=True, stable=True)[:2]
print(f"Median sigma: {toy_sigma:.6f}")
print("Top-2 початкових gains:", one_shot_ids.tolist())
print("MaxHerding з порожнім L:", maxherding(toy_kernel, labeled_ids=[], n_select=2))
print("Новий ID, якщо L = [2]:", maxherding(toy_kernel, labeled_ids=[2], n_select=1))

Median sigma: 1.414214
Top-2 початкових gains: [2, 3]
MaxHerding з порожнім L: [2, 0]
Новий ID, якщо L = [2]: [0]


### UHerding: uncertainty зважує targets

Використовуємо той самий greedy-цикл, але внесок кожного target залежить від його uncertainty:

$$\Delta_U(c)=\sum_{i\in U_0}u_i\max(K_{i,c}-m_i,0).$$

Вага `u_i` належить **рядку**, який кандидат допомагає покрити, а не самому кандидату. Точка з невеликою власною uncertainty теж може бути корисною, якщо представляє багато невизначених сусідів.

`uncertainty: float32 [N]` містить scores у порядку всього pool. Усередині відбору використовуємо лише ваги `U₀`; зміна ваг уже labeled точок не впливає на результат. Ділимо додатні target weights на їхній максимум: спільний множник не змінює argmax, а однакові додатні ваги стають точно одиницями. Якщо всі ваги нульові, всі gains нульові, тому добираємо найменші доступні IDs.

### Adaptive sigma за labeled embeddings

За [лекцією, розділ 5](https://github.com/AI220-Computer-Vision/kse_advanced_computer_vision/blob/main/Module_2/Lecture_2/active_learning_moons.ipynb):

$$\sigma^*=\min_{i,j\in L,\ i\ne j}\|h_i-h_j\|_2.$$

Беремо мінімум лише між різними labeled IDs, без діагоналі. При однакових embeddings застосовуємо `SIGMA_FLOOR`. Якщо labeled зображень менше двох, використовуємо явно передану label-free median sigma всього pool. Це fallback ширини kernel, не заміна Random cold start для методу без навченої моделі.

Відстані кешуємо, але kernel перебудовуємо, коли між раундами змінюється `σ`. Після появи двох labeled точок ця minimum-distance sigma не зростає при розширенні `L` за незмінних embeddings. Усередині одного batch `σ` та uncertainty фіксовані.

У цьому блоці scores синтетичні. Для реальних раундів вони мають надходити з поточного segmentation head; ECE та підбір temperature додамо окремо.

In [10]:
@torch.no_grad()
def adaptive_kernel_sigma(
    distances: Tensor, labeled_ids: list[int], fallback_sigma: float
) -> float:
    """Мінімальна відстань між різними labeled IDs; дані лише з train pool."""
    if any(type(i) is not int or not 0 <= i < len(distances) for i in labeled_ids):
        raise ValueError("L має містити цілі IDs у межах pool")
    if len(set(labeled_ids)) != len(labeled_ids):
        raise ValueError("L не повинно містити повторних IDs")
    if isinstance(fallback_sigma, bool) or not math.isfinite(fallback_sigma) or fallback_sigma <= 0:
        raise ValueError("fallback_sigma має бути скінченним додатним числом")
    if len(labeled_ids) < 2:
        return float(fallback_sigma)

    labeled_distances = distances[labeled_ids][:, labeled_ids]
    rows, columns = torch.triu_indices(
        len(labeled_ids), len(labeled_ids), offset=1, device=distances.device
    )
    minimum = labeled_distances[rows, columns].min().item()
    return max(minimum, SIGMA_FLOOR)


def uherding(
    kernel: Tensor, labeled_ids: list[int], n_select: int, uncertainty: Tensor
) -> list[int]:
    """UHerding використовує uncertainty як ваги target rows."""
    if uncertainty is None:
        raise ValueError("UHerding потребує scores; cold start визначає runner")
    return greedy_kernel_selection(kernel, labeled_ids, n_select, weights=uncertainty)

### Приклад впливу uncertainty

У двох ще не покритих груп однакова геометрія відносно `L`. Ліва група має uncertainty `0.1`, права — `0.9`. MaxHerding вирішує нічию за ID, а UHerding віддає перевагу правій групі.

Для цього порівняння обидва методи отримують **той самий kernel**, щоб ізолювати вплив ваг. У повному benchmark MaxHerding матиме власну median sigma, а UHerding — adaptive sigma.

In [11]:
weighted_embeddings = torch.tensor([
    [1.0, 0.0], [1.0, 0.0],
    [0.0, 1.0], [0.0, -1.0],
    [-1.0, 0.0], [-1.0, 0.0],
])
weighted_labeled_ids = [2, 3]
weighted_scores = torch.tensor([0.1, 0.1, 0.0, 0.0, 0.9, 0.9])
weighted_distances = pairwise_distances(weighted_embeddings)
weighted_sigma = adaptive_kernel_sigma(
    weighted_distances, weighted_labeled_ids,
    fallback_sigma=median_kernel_sigma(weighted_distances),
)
weighted_kernel = gaussian_kernel(weighted_distances, weighted_sigma)

print(f"Adaptive sigma: {weighted_sigma:.6f}")
print("MaxHerding:", maxherding(weighted_kernel, weighted_labeled_ids, 1))
print("UHerding:", uherding(weighted_kernel, weighted_labeled_ids, 1, weighted_scores))

Adaptive sigma: 2.000000
MaxHerding: [0]
UHerding: [4]


### Дві sanity checks з умови

За однакових **додатних** uncertainty weights та однакового kernel UHerding має збігатися з MaxHerding. За дуже малого `σ` і різних embeddings kernel наближається до identity, тому UHerding має збігатися з uncertainty top-B. Для другої перевірки не використовуємо дублікати з попереднього прикладу.

In [12]:
constant_scores = torch.full((6,), 0.7)
assert (
    uherding(weighted_kernel, [2, 3], 2, constant_scores)
    == maxherding(weighted_kernel, [2, 3], 2)
)

local_kernel = gaussian_kernel(pairwise_distances(torch.eye(4)), sigma=1e-8)
local_scores = torch.tensor([0.1, 0.9, 0.5, 0.9])
assert (
    uherding(local_kernel, [0], 2, local_scores)
    == uncertainty_top_b(local_scores, [1, 2, 3], 2)
)
print("Однакові додатні weights → MaxHerding; мале sigma → uncertainty top-B: OK")

Однакові додатні weights → MaxHerding; мале sigma → uncertainty top-B: OK


### Temperature calibration без official val

Temperature scaling перетворює logits на $p_\tau=\operatorname{softmax}(z/\tau)$. Додатна `τ` змінює впевненість, але не порядок класів. Обираємо її за ECE, а не за mIoU на official val.

За лекційною схемою:

1. Поділяємо вже queried `L` на training і calibration **за зображеннями**, не за пікселями.
2. Окремий calibration head навчаємо лише на training subset. Його прогнози на calibration subset використовуємо для підбору `τ`.
3. Основний head навчаємо на всьому `L`. Знайдену `τ` переносимо на його logits перед обчисленням uncertainty.

Це перенесення не гарантує таку саму calibration quality для основного head. Official val залишається лише для reporting. Самі функції нижче не можуть перевірити, на яких зображеннях навчався переданий head; це має забезпечити AL runner.

Для split беремо параметри з конфігурації колеги: 20% labeled зображень, щонайменше 4 для calibration. Має залишитися хоча б одне training image; інакше повертаємо порожній calibration subset і надалі використовуємо `τ=1`. На першому реальному бюджеті 21 це дає 17 training і 4 calibration зображення. IDs сортуємо перед seeded shuffle, щоб порядок попередніх acquisition batches не змінював split.

In [13]:
def split_calibration_ids(
    labeled_ids: list[int],
    seed: int,
    calibration_fraction: float = 0.2,
    min_calibration_images: int = 4,
) -> tuple[list[int], list[int]]:
    """Повертає disjoint training/calibration IDs лише з уже queried L."""
    if any(type(i) is not int or i < 0 for i in labeled_ids):
        raise ValueError("Очікуємо невід'ємні цілі labeled IDs")
    if len(set(labeled_ids)) != len(labeled_ids):
        raise ValueError("L не повинно містити повторних IDs")
    if not 0 < calibration_fraction < 1:
        raise ValueError("calibration_fraction має лежати між 0 та 1")
    if type(min_calibration_images) is not int or min_calibration_images < 1:
        raise ValueError("min_calibration_images має бути додатним цілим числом")
    if type(seed) is not int or not 0 <= seed < 2**63:
        raise ValueError("seed має бути цілим числом у [0, 2**63)")

    ids = sorted(labeled_ids)
    n_calibration = max(min_calibration_images, int(len(ids) * calibration_fraction))
    if n_calibration >= len(ids):
        return ids, []
    generator = torch.Generator().manual_seed(seed)
    order = torch.randperm(len(ids), generator=generator).tolist()
    calibration_ids = sorted(ids[i] for i in order[:n_calibration])
    training_ids = sorted(ids[i] for i in order[n_calibration:])
    return training_ids, calibration_ids

### Pixel-level ECE

Confidence пікселя — найбільша ймовірність класу. Розбиваємо `[0, 1]` на 5 equal-width bins, як у лекції. Межі bins лівозамкнені, останній включає `1.0`.

$$\mathrm{ECE}=\sum_b\frac{n_b}{N}\left|\mathrm{accuracy}_b-\mathrm{confidence}_b\right|.$$

У кожному bin зберігаємо кількість пікселів, суму confidence і кількість правильних прогнозів. Ці суми можна додавати між batches. ECE рахуємо **після** додавання всіх сум, бо середнє batch-wise ECE загалом дає інший результат. Накопичуємо статистику у float64.

Враховуємо лише `valid_pixels & (targets != 255)`. Тут GT masks дозволені, бо це вже queried calibration subset. На unlabeled pool таке фільтрування за GT заборонене.

Кожен валідний піксель має однакову вагу, фон також входить у ECE. Велика частка фону може приховувати погану calibration рідкісних класів. За відсутності валідних пікселів ECE невизначене, а не нульове.

In [14]:
@torch.no_grad()
def ece_bin_sums(
    logits: Tensor,
    targets: Tensor,
    valid_pixels: Tensor,
    temperature: float = 1.0,
    n_bins: int = ECE_BINS,
) -> Tensor:
    """Повертає [3, n_bins]: counts, confidence sums, correct counts."""
    if logits.ndim != 4 or min(logits.shape) < 1 or logits.shape[1] < 2:
        raise ValueError("Очікуємо непорожні logits [B, C, H, W], C >= 2")
    if targets.shape != logits.shape[:1] + logits.shape[2:] or valid_pixels.shape != targets.shape:
        raise ValueError("Targets і geometry mask повинні мати форму [B, H, W]")
    if logits.dtype != torch.float32 or targets.dtype != torch.int64 or valid_pixels.dtype != torch.bool:
        raise ValueError("Очікуємо float32 logits, int64 targets та bool geometry mask")
    if targets.device != logits.device or valid_pixels.device != logits.device:
        raise ValueError("Усі tensors мають бути на одному device")
    if not torch.isfinite(logits).all():
        raise ValueError("Logits мають бути скінченними")
    if isinstance(temperature, bool) or not math.isfinite(temperature) or temperature <= 0:
        raise ValueError("Temperature має бути скінченним додатним числом")
    if type(n_bins) is not int or n_bins < 1:
        raise ValueError("n_bins має бути додатним цілим числом")

    keep = valid_pixels & (targets != IGNORE_INDEX)
    labels = targets[keep]
    if ((labels < 0) | (labels >= logits.shape[1])).any():
        raise ValueError("Calibration targets містять невідомий клас")
    confidence, predicted = (logits / temperature).softmax(dim=1).max(dim=1)
    confidence = confidence[keep].double()
    correct = (predicted[keep] == labels).double()
    bin_ids = (confidence * n_bins).long().clamp_max(n_bins - 1)
    return torch.stack([
        torch.bincount(bin_ids, minlength=n_bins).double(),
        torch.bincount(bin_ids, weights=confidence, minlength=n_bins).double(),
        torch.bincount(bin_ids, weights=correct, minlength=n_bins).double(),
    ])


def expected_calibration_error(bin_sums: Tensor) -> float:
    """ECE зі спільних bin sums усіх calibration batches."""
    n_pixels = bin_sums[0].sum()
    if n_pixels <= 0:
        raise ValueError("ECE невизначене без валідних calibration pixels")
    # n_b * |mean(confidence) - mean(correct)| = |sum(confidence) - sum(correct)|.
    return ((bin_sums[1] - bin_sums[2]).abs().sum() / n_pixels).item()

### Пошук temperature

Беремо лекційну сітку `0.25, 0.5, 1, 2, 4, 8`, зафіксовану до експериментів. Для кожної `τ` накопичуємо bin sums на всьому calibration subset і вибираємо найменше ECE. При точній нічиїй перемагає перше значення в сітці.

`fit_temperature` приймає iterator із tuples `(logits, targets, valid_pixels)`. Проходимо його один раз: inference не потрібно повторювати для кожної temperature, і всі calibration logits не треба тримати в RAM. Зберігаємо лише статистику bins та поточний batch.

Порожній iterator або відсутність валідних пікселів дають явний fallback `(1.0, {})`. Порожній словник означає, що ECE оцінити не вдалося. Для реальних даних tuples складатимемо з `CalibrationBatch` лише після перевірки походження його IDs.

In [15]:
@torch.no_grad()
def fit_temperature(
    calibration_batches: Iterable[tuple[Tensor, Tensor, Tensor]],
    temperatures: tuple[float, ...] = TEMPERATURE_GRID,
    n_bins: int = ECE_BINS,
) -> tuple[float, dict[float, float]]:
    """Повертає найкращу tau та ECE для кожної temperature на held-out L."""
    if not temperatures or any(
        isinstance(t, bool) or not math.isfinite(t) or t <= 0 for t in temperatures
    ):
        raise ValueError("Потрібна непорожня сітка скінченних додатних temperatures")
    if len(set(temperatures)) != len(temperatures):
        raise ValueError("Temperature grid не повинна містити повторів")
    if type(n_bins) is not int or n_bins < 1:
        raise ValueError("n_bins має бути додатним цілим числом")

    totals = torch.zeros(len(temperatures), 3, n_bins, dtype=torch.float64)
    for logits, targets, valid_pixels in calibration_batches:
        for index, temperature in enumerate(temperatures):
            totals[index] += ece_bin_sums(
                logits, targets, valid_pixels, temperature, n_bins
            ).cpu()

    if totals[0, 0].sum() == 0:
        return 1.0, {}
    errors = {
        float(temperature): expected_calibration_error(totals[index])
        for index, temperature in enumerate(temperatures)
    }
    best_temperature = min(errors, key=lambda temperature: errors[temperature])
    return best_temperature, errors

### Синтетичний приклад надмірної впевненості

Нижче окремо перевіряємо split для бюджету 21 і calibration на чотирьох штучних пікселях. Head не навчаємо. Для кожного пікселя модель дає класу `0` імовірність `0.99`, але правильними є лише три прогнози з чотирьох.

При `τ=1` ECE дорівнює приблизно `|0.99 - 0.75| = 0.24`. Більша temperature зменшить confidence, не змінюючи передбачений клас. Поліпшення ECE на цьому прикладі не означає поліпшення segmentation accuracy або гарантії на реальних даних.

In [16]:
fit_ids, calibration_ids = split_calibration_ids(list(range(21)), seed=SEED + 20_000)
print(f"Бюджет 21: training={len(fit_ids)}, calibration={len(calibration_ids)}")

calibration_probabilities = torch.full((NUM_CLASSES,), 0.01 / (NUM_CLASSES - 1))
calibration_probabilities[0] = 0.99
calibration_logits = calibration_probabilities.log()[None, :, None, None].expand(2, -1, 1, 2)
calibration_targets = torch.tensor([[[0, 0]], [[0, 1]]])
calibration_valid_pixels = torch.ones_like(calibration_targets, dtype=torch.bool)
calibration_batches = (
    (calibration_logits[i:i + 1], calibration_targets[i:i + 1], calibration_valid_pixels[i:i + 1])
    for i in range(2)
)
temperature, calibration_errors = fit_temperature(calibration_batches)
print(f"Обрана temperature: {temperature:g}")
print(f"ECE до calibration: {calibration_errors[1.0]:.6f}")
print(f"ECE після calibration: {calibration_errors[temperature]:.6f}")

# На реальному pool так само масштабуємо logits, але жодних GT masks не читаємо.
calibrated_pixel_scores = pixel_uncertainty(calibration_logits / temperature, method="margin")

Бюджет 21: training=17, calibration=4
Обрана temperature: 2
ECE до calibration: 0.239999
ECE після calibration: 0.060091


## Інтеграція з `alseg`

Використовуємо код колеги для `Oracle`, extraction, навчання нового head та evaluation. Наші функції відповідають за acquisition і calibration. `src/` не переписуємо.

Поруч із notebook потрібен каталог `src/source/alseg`. Для цього розділу додатково потрібні `Pillow` та `lightning==2.6.6`. Автоматичного встановлення, завантаження VOC або навчання при виконанні notebook немає: нижче визначено функції, а реальний запуск буде окремим викликом.

`NotebookSelector` реалізує наявний інтерфейс `Selector.select(SelectionRequest)`. Він кешує відстані для одного незмінного набору embeddings і не отримує доступу до Oracle або evaluation split.

In [17]:
import json
import sys
import time
from pathlib import Path

SOURCE_DIR = Path("src/source").resolve()
if not (SOURCE_DIR / "alseg").is_dir():
    raise FileNotFoundError("Потрібен каталог src/source/alseg поруч із notebook")
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

from alseg.core.calibration import CalibrationBatch
from alseg.core.helpers import write_csv
from alseg.core.predictions import LogitBatch
from alseg.core.scores import assemble_scores
from alseg.core.selection import AcquisitionMethod, SelectionRequest, validate_selection
from alseg.data.preprocessing import resize_mask
from alseg.interfaces.acquisition import Selector
from alseg.interfaces.data import EvaluationSource, ImageSource, Oracle
from alseg.interfaces.encoder import Encoder
from alseg.nn.evaluation import evaluate_head
from alseg.nn.extraction import iter_features
from alseg.nn.training import fit_head

In [18]:
class NotebookSelector(Selector):
    """Acquisition для одного frozen encoder і незмінного порядку pool IDs."""

    def __init__(self, embeddings: Tensor):
        self.embeddings = embeddings.detach().clone()
        self.distances = pairwise_distances(self.embeddings)
        self.median_sigma = (
            median_kernel_sigma(self.distances) if len(embeddings) > 1 else 1.0
        )

    def select(self, request: SelectionRequest) -> list[int]:
        if not torch.equal(request.embeddings, self.embeddings):
            raise ValueError("Embeddings не відповідають кешованим відстаням selector")
        if request.method == AcquisitionMethod.RANDOM:
            generator = torch.Generator().manual_seed(request.seed)
            order = torch.randperm(len(request.unlabeled_ids), generator=generator)
            selected = [request.unlabeled_ids[i] for i in order[:request.n_select].tolist()]
        elif request.method == AcquisitionMethod.UNCERTAINTY:
            selected = uncertainty_top_b(
                request.uncertainty, request.unlabeled_ids, request.n_select
            )
        else:
            kernel = gaussian_kernel(self.distances, request.sigma)
            if request.method == AcquisitionMethod.MAXHERDING:
                selected = maxherding(kernel, request.labeled_ids, request.n_select)
            else:
                selected = uherding(
                    kernel, request.labeled_ids, request.n_select, request.uncertainty
                )
        validate_selection(request, selected)
        return selected

### Передавання даних між компонентами

Pool inference отримує лише `ImageSource`, encoder і head. Йому не передаємо Oracle. Маски читаємо окремо, лише для вже queried training/calibration IDs.

Для training і acquisition використовуємо квадратну роздільність `config["encoder"]["image_size"]`, як у preprocessing колеги. Кадр масштабується цілком, без padding, тому geometry mask складається з `True`. Official val оцінює функція колеги в початкових розмірах масок.

Ця перша інтеграція повторно витягує patch features через `iter_features`, як наявний pipeline. У RAM залишаються features поточного `L`, а не всього pool; logits обробляємо батчами. Постійний patch cache для великого benchmark поки не додаємо.

In [19]:
def extract_pool_embeddings(pool: ImageSource, encoder: Encoder, batch_size: int) -> Tensor:
    """Embeddings у канонічному порядку всіх train pool IDs, без масок."""
    return torch.cat([
        features.embeddings
        for _, features in iter_features(pool, encoder, list(range(len(pool))), batch_size)
    ])


def prepare_labeled_training(
    pool: ImageSource, oracle: Oracle, encoder: Encoder,
    labeled_ids: list[int], image_size: int, batch_size: int,
) -> tuple[Tensor, Tensor]:
    """Ті самі IDs і порядок для patch features та вже queried masks."""
    features = torch.cat([
        batch.patch_features
        for _, batch in iter_features(pool, encoder, labeled_ids, batch_size)
    ])
    targets = torch.stack([
        resize_mask(oracle.get_mask(image_id), image_size) for image_id in labeled_ids
    ])
    return features, targets


@torch.no_grad()
def iter_model_logits(
    head: torch.nn.Module, pool: ImageSource, encoder: Encoder,
    ids: list[int], image_size: int, batch_size: int, device: str,
) -> Iterable[LogitBatch]:
    """Потоковий inference без масок; після завершення head повертається на CPU."""
    head.to(device).eval()
    try:
        for batch_ids, features in iter_features(pool, encoder, ids, batch_size):
            logits = head(features.patch_features.to(device), (image_size, image_size)).cpu()
            valid_pixels = torch.ones_like(logits[:, 0], dtype=torch.bool)
            yield LogitBatch(batch_ids, logits, valid_pixels)
    finally:
        head.cpu()


def calibration_batches_from_logits(
    predictions: Iterable[LogitBatch], oracle: Oracle,
) -> Iterable[tuple[Tensor, Tensor, Tensor]]:
    """Лише прогнози на held-out queried IDs, визначених runner."""
    for batch in predictions:
        targets = torch.stack([
            resize_mask(oracle.get_mask(image_id), batch.logits.shape[-1])
            for image_id in batch.ids
        ])
        calibration = CalibrationBatch(batch, targets)
        yield calibration.predictions.logits, calibration.targets, batch.valid_pixels

### Послідовність AL-раунду

`run_active_learning` приймає компоненти колеги та його config dict. Методи: `random`, `uncertainty`, `maxherding`, `uherding`. `config["budgets"]` задає накопичені бюджети; selector отримує лише різницю між новим бюджетом і поточним `|L|`.

Для uncertainty та UHerding використовуємо однаковий cold-start fallback із config. Починаючи з другого раунду, обидва отримують calibrated scores поточного head. Окремий calibration head навчається на subset попереднього `L`; основний head після нового query навчається з нуля на всьому розширеному `L`.

Seed ініціалізації head однаковий для всіх методів і раундів. Seed відбору змінюється за номером раунду, але не за методом. Official val читає лише `evaluate_head`, і його метрики не передаються в acquisition.

Кожен раунд зберігає checkpoint, history, metrics та acquisition/calibration IDs. CSV містить `acquisition_budget` для sigma/tau, використаних **до** нового query, та `budget` для mIoU **після** навчання. Resume поки не підтримуємо: потрібен свіжий Oracle; наявний `config.json` не перезаписуємо.

In [20]:
def run_active_learning(
    pool: ImageSource,
    oracle: Oracle,
    encoder: Encoder,
    evaluation: EvaluationSource,
    config: dict,
    method: str,
    run_dir: Path,
    device: str = "cpu",
    embeddings: Tensor | None = None,
    smoke: bool = False,
) -> list[dict]:
    """Повний AL-цикл; calibration використовує попередній L, не official val."""
    method = AcquisitionMethod(method)
    budgets = list(config["budgets"])
    if not budgets or any(type(b) is not int or not 0 < b <= len(pool) for b in budgets):
        raise ValueError("Бюджети мають бути додатними цілими числами в межах pool")
    if budgets != sorted(set(budgets)) or oracle.labeled_ids:
        raise ValueError("Потрібні строго зростаючі бюджети та свіжий Oracle")
    if device not in ("cpu", "cuda") or (device == "cuda" and not torch.cuda.is_available()):
        raise ValueError("Запитаний training device недоступний")
    if len(evaluation) == 0:
        raise ValueError("Evaluation split не може бути порожнім")
    if config["data"]["num_classes"] != NUM_CLASSES or config["data"]["ignore_index"] != IGNORE_INDEX:
        raise ValueError("Потрібні 21 клас і ignore_index=255")
    if config["task"].get("head", "linear_1x1") != "linear_1x1" or not config["task"].get("retrain_from_scratch", True):
        raise ValueError("Цей runner використовує свіжий linear_1x1 head у кожному раунді")

    protocol = config["protocol"]
    cold_start = AcquisitionMethod(protocol["cold_start"])
    if cold_start not in (AcquisitionMethod.RANDOM, AcquisitionMethod.MAXHERDING):
        raise ValueError("Cold start має бути random або maxherding")
    image_size = config["encoder"]["image_size"]
    batch_size = config["encoder"]["batch_size"]
    training_seed = config["seed"] + protocol["training_seed_offset"]
    calibration_seed = config["seed"] + protocol["calibration_seed_offset"]
    uncertainty_method = config["acquisition"].get("uncertainty_method", "margin")
    temperatures = tuple(config["acquisition"].get("temperatures", TEMPERATURE_GRID))
    n_bins = config["acquisition"].get("ece_bins", ECE_BINS)
    model_based = method in (AcquisitionMethod.UNCERTAINTY, AcquisitionMethod.UHERDING)

    run_dir = Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)
    run_config = {
        **config, "method": method.value, "smoke": smoke, "device": device,
        "encoder_identity": encoder.describe(), "n_pool": len(pool), "n_val": len(evaluation),
        "uncertainty_method": uncertainty_method, "temperatures": temperatures, "ece_bins": n_bins,
    }
    with (run_dir / "config.json").open("x") as file:
        json.dump(run_config, file, indent=2, allow_nan=False)
    if embeddings is None:
        embeddings = extract_pool_embeddings(pool, encoder, batch_size)
    if len(embeddings) != len(pool):
        raise ValueError("Embeddings мають покривати весь pool у канонічному порядку")
    selector = NotebookSelector(embeddings)
    head, features, targets = None, None, None
    records = []

    for round_id, budget in enumerate(budgets):
        started = time.perf_counter()
        previous_ids = sorted(oracle.labeled_ids)
        selected_method = cold_start if model_based and head is None else method
        temperature, errors, scores, sigma = None, {}, None, None
        calibration_fit_ids, calibration_ids = [], []
        if model_based and head is not None:
            calibration_fit_ids, calibration_ids = split_calibration_ids(
                previous_ids, calibration_seed,
                protocol["calibration_fraction"], protocol["calibration_min_images"],
            )
            temperature = 1.0
            if calibration_ids:
                positions = {image_id: row for row, image_id in enumerate(previous_ids)}
                fit_rows = [positions[i] for i in calibration_fit_ids]
                calibration_task = fit_head(
                    features[fit_rows], targets[fit_rows], config["task"],
                    training_seed, device, logger=False,
                )
                predictions = iter_model_logits(
                    calibration_task.head, pool, encoder, calibration_ids,
                    image_size, batch_size, device,
                )
                temperature, errors = fit_temperature(
                    calibration_batches_from_logits(predictions, oracle), temperatures, n_bins
                )
                del calibration_task
            score_batches = [
                (batch.ids, mean_image_uncertainty(
                    pixel_uncertainty(batch.logits / temperature, uncertainty_method), batch.valid_pixels
                ))
                for batch in iter_model_logits(
                    head, pool, encoder, list(range(len(pool))), image_size, batch_size, device
                )
            ]
            scores = assemble_scores(len(pool), score_batches)
        if selected_method == AcquisitionMethod.MAXHERDING:
            sigma = selector.median_sigma
        elif selected_method == AcquisitionMethod.UHERDING:
            sigma = adaptive_kernel_sigma(selector.distances, previous_ids, selector.median_sigma)

        request = SelectionRequest(
            method=selected_method, embeddings=embeddings,
            labeled_ids=previous_ids,
            unlabeled_ids=sorted(set(range(len(pool))) - set(previous_ids)),
            n_select=budget - len(previous_ids),
            seed=config["seed"] + protocol["selection_seed_offset"] + round_id,
            uncertainty=scores, sigma=sigma,
        )
        selected_ids = selector.select(request)
        acquisition_seconds = time.perf_counter() - started
        oracle.query(selected_ids)
        labeled_ids = sorted(oracle.labeled_ids)

        started = time.perf_counter()
        features, targets = prepare_labeled_training(
            pool, oracle, encoder, labeled_ids, image_size, batch_size
        )
        task = fit_head(features, targets, config["task"], training_seed, device, logger=False)
        head = task.head
        training_seconds = time.perf_counter() - started
        started = time.perf_counter()
        metrics, _ = evaluate_head(
            head, encoder, evaluation, list(range(len(evaluation))), batch_size, device
        )
        evaluation_seconds = time.perf_counter() - started

        round_dir = run_dir / f"budget_{budget:04d}"
        round_dir.mkdir(exist_ok=False)
        acquisition = {
            "selected_ids": selected_ids, "labeled_ids": labeled_ids,
            "selection_method": selected_method.value, "selection_seed": request.seed,
            "acquisition_budget": len(previous_ids), "budget": budget,
            "calibration_fit_ids": calibration_fit_ids, "calibration_ids": calibration_ids,
            "tau": temperature, "ece_by_temperature": errors, "sigma": sigma,
        }
        (round_dir / "acquisition.json").write_text(json.dumps(acquisition, indent=2, allow_nan=False))
        (round_dir / "metrics.json").write_text(json.dumps(metrics, indent=2, allow_nan=False))
        write_csv(round_dir / "training.csv", task.history)
        torch.save({
            "state_dict": head.state_dict(), "feature_dim": features.shape[1],
            "encoder": encoder.describe(), "labeled_ids": labeled_ids, "training_seed": training_seed,
        }, round_dir / "head.pt")
        del task
        records.append({
            "round": round_id, "method": method.value, "selection_method": selected_method.value,
            "acquisition_budget": len(previous_ids), "budget": budget,
            "n_calibration": len(calibration_ids), "tau": temperature, "sigma": sigma,
            "miou": metrics["miou"], "pixel_accuracy": metrics["pixel_accuracy"],
            "mean_class_accuracy": metrics["mean_class_accuracy"], "val_loss": metrics["loss"],
            "acquisition_seconds": acquisition_seconds, "training_seconds": training_seconds,
            "evaluation_seconds": evaluation_seconds, "smoke": smoke,
        })
        write_csv(run_dir / "results.csv", records)
    return records

### Перевірки та наступний крок

Детальні тести лежать у `tests/`. Запуск із кореня проєкту: `python -m unittest discover -s tests -v`. Unit tests перевіряють алгоритми без навчання. `test_integration.py` створює тимчасові VOC-подібні файли й запускає чотири методи на CPU з бюджетами 21 і 42 та однією training epoch. Він використовує справжні Oracle, training та evaluation з `alseg`, але штучні зображення й encoder. Завантажень і зовнішнього логування немає.

Інтеграційні перевірки охоплюють відсутність повторних queries, спільний cold start, disjoint calibration split, відповідність features і masks тим самим IDs, новий основний head на всьому `L`, окремий evaluation split та збереження результатів. Синтетичні метрики не є benchmark якості сегментації.

У notebook залишено короткі приклади та дві обов'язкові sanity checks UHerding. Виконання notebook згори донизу визначає AL runner, але не запускає training.

Наступний крок — Colab, VOC і реальний frozen encoder. Перед повним benchmark потрібен обмежений GPU smoke test із вимірюванням часу та пікової пам'яті. Постійного patch cache, resume та full-data upper bound у цьому runner поки немає.